In [3]:
import pandas as pd
import tensorflow.keras.layers as Layers
import tensorflow.keras.activations as Actications
import tensorflow.keras.models as Models
import tensorflow.keras.optimizers as Optimizer
import tensorflow.keras.metrics as Metrics
import tensorflow.keras.utils as Utils
#from keras.utils.vis_utils import model_to_dot
import os
import matplotlib.pyplot as plot
import cv2
import numpy as np
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix as CM
from random import randint
#from IPython.display import SVG
import matplotlib.gridspec as gridspec
from tensorflow.keras.models import model_from_json
import pathlib
import re
from dataclasses import dataclass
from typing import Dict
import pickle


<b>Classify Head/Not Head</b>
Import head classifier model

In [4]:
#
input_folders = "/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/"
output_path = "/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir"
####
cohort_list = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/XRY normal skull WT animal.xlsx"


Scripts for parsing normal subjects

In [5]:
@dataclass
class MouseDescriptor:

    cohort_id: str
    gender: str
    date: str
    subj_id: str
    subj_fold: pathlib.Path
    cohort_fold: pathlib.Path
    preprocessed: bool

class Subject:

    def __init__(self,subf):
        self.specimen = subf['SPECIMEN_ID']
        self.cohort = subf['COHORT_ID']
        self.gender = str(subf['GENDER'])
        self.date = subf['DATE']
        self.scanner = subf['Machine']

In [6]:


dataset : Dict[str, MouseDescriptor] = {}

def has_head(path: pathlib.Path) -> bool:
    """
    return
    :param path: path of folder
    :return: True if folder have been processed
    """

    if len(list(path.glob("**/*__head"))) > 0:
        return True
    return False


cohorts = list( pd.read_excel(cohort_list) ['COHORT_ID'])
cohort_exel = pd.read_excel(cohort_list)
import tqdm.auto
proc_files = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/scripts/snakemake_workflows/processed_files.pkl"
print("Starting processing files")
if os.path.exists(proc_files):
    f = open(proc_files,"rb")
    dataset = pickle.load(f)
    f.close()
else:
    for fold_ in tqdm.tqdm(list(pathlib.Path(input_folders).iterdir())):
        print(str(fold_))
        if fold_.is_file():
            continue
        fn_ =fold_.name
        #print(fn_)
        match = re.search(r'c\d+', fn_)
        if match:
            res = match.group()
            print(res)
            if res in cohorts:
                indexes = list(cohort_exel[cohort_exel['COHORT_ID'] == res].index)
                ####generate Descriptors
                folers = list(fold_.iterdir()) # folders inside cohort
                for id in indexes:
                    subj = Subject(cohort_exel.loc[id])
                    #get folder with subject
                    #print(cohort_exel.loc[id])
                    #print(fold_)
                    print("processing cohort " + subj.cohort + " and specimen " + subj.specimen )
                    subj_folds = [x for x in folers if x.name.endswith(subj.specimen)]
                    if len(subj_folds) > 0:
                        subj_fold = subj_folds[0]
                        dataset[subj_fold] = MouseDescriptor(subj.cohort,subj.gender,subj.date,
                                                             subj_fold=subj_fold,cohort_fold=fold_,
                                                             preprocessed=has_head(subj_fold),subj_id=subj.specimen)
                        print("Found and processed")
                    if len(subj_folds) > 1:
                        print(subj_folds)
            else:
                print("pattern error? "  + res + " fn " + fn_)
    f = open(proc_files,"wb")
    pickle.dump(dataset,f)
    f.close()

Starting processing files


  0%|          | 0/340 [00:00<?, ?it/s]

/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20220321c231_Ccdc198 & Myo1g & Rras2 & wt & Zc3h8
c231
processing cohort c231 and specimen 82Z-09479
Found and processed
processing cohort c231 and specimen 82Z-09480
Found and processed
processing cohort c231 and specimen 82Z-09481
Found and processed
processing cohort c231 and specimen 82Z-09476
Found and processed
processing cohort c231 and specimen 82Z-09477
Found and processed
processing cohort c231 and specimen 82Z-09478


  0%|          | 1/340 [00:24<2:17:33, 24.35s/it]

Found and processed
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/DATA Bioimaging UCT.xlsx
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20201228c171_Prss21 & Prss35 & wt
c171
processing cohort c171 and specimen 82Z-05335
Found and processed
processing cohort c171 and specimen 82Z-05336
Found and processed
processing cohort c171 and specimen 82Z-05337
Found and processed
processing cohort c171 and specimen 82Z-05343
Found and processed
processing cohort c171 and specimen 82Z-05344
Found and processed
processing cohort c171 and specimen 82Z-05345
Found and processed
processing cohort c171 and specimen 82Z-05346
Found and processed
processing cohort c171 and specimen 82Z-05326
Found and processed
processing cohort c171 and specimen 82Z-05327
Found and processed
processing cohort c171 and specimen 82Z-05328
Found and processed
processing cohort c171 and specimen 82Z-05338
Found and processed
processing cohort c171 and specimen 

...

Found and processed
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20201221c170_Klhl1 & Meis3 & wt & Zc1HC1c
c170
processing cohort c170 and specimen 82Z-05180
Found and processed
processing cohort c170 and specimen 82Z-05197
Found and processed
processing cohort c170 and specimen 82Z-05198
Found and processed
processing cohort c170 and specimen 82Z-05199
Found and processed
processing cohort c170 and specimen 82Z-05218
Found and processed
processing cohort c170 and specimen 82Z-05219
Found and processed
processing cohort c170 and specimen 82Z-05178
Found and processed
processing cohort c170 and specimen 82Z-05179
Found and processed
processing cohort c170 and specimen 82Z-05214
Found and processed
processing cohort c170 and specimen 82Z-05215
Found and processed
processing cohort c170 and specimen 82Z-05216
Found and processed
processing cohort c170 and specimen 82Z-05217


 99%|█████████▉| 336/340 [3:42:42<01:31, 22.80s/it]

Found and processed
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20210118c174_Btbd18 & Klhl6 & wt
c174
processing cohort c174 and specimen 82Z-05643
Found and processed
processing cohort c174 and specimen 82Z-05644
Found and processed
processing cohort c174 and specimen 82Z-05650
Found and processed
processing cohort c174 and specimen 82Z-05651
Found and processed
processing cohort c174 and specimen 82Z-05652
Found and processed
processing cohort c174 and specimen 82Z-CD920
Found and processed
processing cohort c174 and specimen 82Z-CD921
Found and processed
processing cohort c174 and specimen 82Z-05624
Found and processed
processing cohort c174 and specimen 82Z-05626
Found and processed
processing cohort c174 and specimen 82Z-05629
Found and processed
processing cohort c174 and specimen 82Z-05630
Found and processed
processing cohort c174 and specimen 82Z-05631
Found and processed
processing cohort c174 and specimen 82Z-05632
Found and processed
proc

100%|██████████| 340/340 [3:43:12<00:00, 39.39s/it]

Found and processed
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/DeadPixelDetection.pptx
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20170123_Abcg8 & WT(C57Bl-6NCrl)
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/matched_dirs.txt


In [9]:
from configparser import ConfigParser
from typing import Union, List, Dict
from PIL import Image
import SimpleITK as sitk
# deve
def get_subjects(fold : Union[str,pathlib.Path]) -> List[pathlib.Path]:
    """
    parse folder and get all mice folders inside
    :param fold: folder with cohort
    :return: folder with mouse
    """
    i_f = fold
    if isinstance(fold,str):
        i_f = pathlib.Path(i_f)
    res = []
    for file_ in i_f.iterdir():
        if file_.is_file():
            continue
        #filter
        if file_.name.startswith('BMD_TMD') or file_.name.startswith('Morphology') or file_.name.startswith('phantom'):
            continue
        res.append(file_)
    return res


class ImageProperties:
    def __init__(self, ini_file):
        config = ConfigParser()
        config.read(ini_file,encoding='cp1252')

        # System
        self.scanner = config.get('System', 'Scanner')
        self.instrument_sn = config.get('System', 'Instrument S/N')
        self.home_directory = config.get('System', 'Home Directory')
        self.source_type = config.get('System', 'Source Type')
        self.camera_pixel_size = float(config.get('System', 'Camera Pixel Size (um)'))

        # Acquisition
        self.data_directory = config.get('Acquisition', 'Data Directory')
        self.filename_prefix = config.get('Acquisition', 'Filename Prefix')
        self.number_of_files = int(config.get('Acquisition', 'Number Of Files'))
        self.number_of_rows = int(config.get('Acquisition', 'Number Of Rows'))
        self.number_of_columns = int(config.get('Acquisition', 'Number Of Columns'))
        self.filename_index_length = int(config.get('Acquisition', 'Filename Index Length'))
        #self.frame_index_length = int(config.get('Acquisition', 'Frame Index Length'))
        #self.partial_width = config.get('Acquisition', 'Partial Width')
        self.image_crop_origin_x = int(config.get('Acquisition', 'Image crop origin X'))
        self.image_crop_origin_y = int(config.get('Acquisition', 'Image crop origin Y'))
        self.camera_binning = config.get('Acquisition', 'Camera binning')
        self.image_rotation = float(config.get('Acquisition', 'Image Rotation'))
        self.optical_axis = int(config.get('Acquisition', 'Optical Axis (line)'))
        self.object_to_source = float(config.get('Acquisition', 'Object to Source (mm)'))
        self.camera_to_source = float(config.get('Acquisition', 'Camera to Source (mm)'))
        self.source_voltage = int(config.get('Acquisition', 'Source Voltage (kV)'))
        self.source_current = int(config.get('Acquisition', 'Source Current (uA)'))
        self.image_pixel_size = float(config.get('Acquisition', 'Image Pixel Size (um)'))
        #self.scaled_image_pixel_size = float(config.get('Acquisition', 'Scaled Image Pixel Size (um)'))
        self.image_format = config.get('Acquisition', 'Image Format')
        self.depth = int(config.get('Acquisition', 'Depth (bits)'))
        self.exposure = int(config.get('Acquisition', 'Exposure (ms)'))
        self.rotation_step = float(config.get('Acquisition', 'Rotation Step (deg)'))
        self.use_360_rotation = config.get('Acquisition', 'Use 360 Rotation')
        #self.scanning_position = float(config.get('Acquisition', 'Scanning position'))
        #self.frame_averaging = config.get('Acquisition', 'Frame Averaging')
        #self.flat_field_correction = config.get('Acquisition', 'Flat Field Correction')
        #self.geometrical_correction = config.get('Acquisition', 'Geometrical Correction')
        self.filter = config.get('Acquisition', 'Filter')



def convert_dataset_to_image(dataset: Dict[int,np.ndarray],image_props:ImageProperties):

    #check missing
    arr  = sorted(dataset)
    for i in range(len(sorted(arr))-1):
        if arr[i+1] - arr[i] > 1:
            print("Missing values between", arr[i], "and", arr[i+1])
            raise Exception()

    res_image = []
    for id in sorted(dataset):
        res_image.append( dataset[id] )

    res_array = np.array(res_image)
    affine_m = np.eye(4)
    affine_m[0,0] = affine_m[1,1] = affine_m[2,2] = image_props.image_pixel_size

    res_im = sitk.GetImageFromArray(res_array)
    voxel_size = (image_props.image_pixel_size, image_props.image_pixel_size, image_props.image_pixel_size)
    #res_im.SetDirection(affine_m.flatten())
    res_im.SetSpacing(voxel_size)
    return res_im


def convert_head_folder(fold : Union[str,pathlib.Path]):
    """

    :param fold: folder with a head
    :return:
    """
    i_f = fold
    if isinstance(fold,str):
        i_f = pathlib.Path(i_f)
    #folder already preprocessed
    rec_folder = [ x for x in i_f.iterdir() if x.is_dir() and x.name.endswith("Rec")][0]

    head_folder = list(rec_folder.glob("**/*__head"))[0]

    config_file = [x for x in head_folder.iterdir() if x.name.endswith(".log")][0]
    config = ImageProperties(config_file)

    #image files
    dataset = {}
    for image_file in head_folder.glob("**/*.bmp"):
        #extra filter of head folder
        if image_file.name.endswith("spr.bmp"):
            continue
        ids = (int(image_file.name.replace(".bmp",'').split('rec')[-1]))
        arr = np.array(Image.open(image_file))
        dataset[ids] = arr

    return convert_dataset_to_image(dataset,config)


    #for
    #print(config.image_pixel_size)




In [13]:
# write images to output
res = []
for key, data in tqdm.tqdm(dataset.items()):

    if not data.preprocessed:
        continue
    id_name = pathlib.Path(key).name
    head_f = None
    for x in data.subj_fold.glob("**/*_head"):
        head_f = x
        break
    res.append([data.subj_id, data.cohort_id,len(list(head_f.iterdir()))])



100%|██████████| 1873/1873 [24:05<00:00,  1.30it/s]  


In [14]:
res

[['82Z-09479', 'c231', 511],
 ['82Z-09480', 'c231', 501],
 ['82Z-09481', 'c231', 514],
 ['82Z-09476', 'c231', 530],
 ['82Z-09477', 'c231', 493],
 ['82Z-09478', 'c231', 513],
 ['82Z-10394', 'c245', 562],
 ['82Z-10395', 'c245', 536],
 ['82Z-10396', 'c245', 541],
 ['82Z-10391', 'c245', 513],
 ['82Z-10392', 'c245', 571],
 ['82Z-10393', 'c245', 511],
 ['82Z-10876', 'c253', 552],
 ['82Z-10879', 'c253', 568],
 ['82Z-10873', 'c253', 556],
 ['82Z-10874', 'c253', 557],
 ['82Z-10875', 'c253', 533],
 ['82Z-10878', 'c253', 557],
 ['82Z-09722', 'c235', 542],
 ['82Z-09723', 'c235', 534],
 ['82Z-CF802', 'c235', 544],
 ['82Z-CF808', 'c235', 571],
 ['82Z-09736', 'c235', 552],
 ['82Z-09737', 'c235', 544],
 ['82Z-09738', 'c235', 523],
 ['82Z-09739', 'c235', 519],
 ['82Z-CF805', 'c235', 532],
 ['82Z-CF807', 'c235', 61],
 ['82Z-CG966', 'c264', 529],
 ['82Z-CG967', 'c264', 541],
 ['82Z-CG968', 'c264', 563],
 ['82Z-CG969', 'c264', 530],
 ['82Z-CG970', 'c264', 533],
 ['82Z-CG972', 'c264', 553],
 ['82Z-CG963', 

In [15]:
df = pd.DataFrame(res)

In [21]:
df[df[2]>700]

,0,1,2
214,82Z-11086,c258,734
218,82Z-11212,c261,1547
219,82Z-11213,c261,1372
379,82Z-CE727,c199,1379
403,82Z-11935,c273,1318
405,82Z-11937,c273,1354
435,82Z-10642,c249,1350
479,82Z-11311,c262,779
568,82Z-CG579,c254,1595
631,82Z-07341,c198,1372
